In [1]:
import sys

sys.path.insert(0, "../src")

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [45]:
from datetime import datetime, timezone
from pathlib import Path

from marketforge.acquisition.client import MarketForgeClient
from marketforge.models import (
    AcquisitionRequest,
    DataType,
    Exchange,
    Instrument,
    InstrumentType,
    MarketCategory,
)


request = AcquisitionRequest(
    target=Instrument(
        exchange=Exchange.BYBIT,
        instrument_type=InstrumentType.PERPETUAL,
        market_category=MarketCategory.LINEAR,
        symbol="BTCUSDT",
    ),
    data_type=DataType.TRADE_TICKS,
    start=datetime(
        2026,
        9,
        1,
        tzinfo=timezone.utc,
    ),
    end=datetime(
        2026,
        9,
        2,
        tzinfo=timezone.utc,
    ),
)


with MarketForgeClient(
    data_root=Path("data")
) as client:
    plan = client.plan(request)

    print("Files    :", plan.file_count)
    print("Download :", plan.download_count)
    print("Reuse    :", plan.reuse_count)
    print()

    for item in plan.downloads:
        print(
            f"{item.action.value:<10} "
            f"{item.destination}"
        )

Files    : 1
Download : 0
Reuse    : 1

reuse      data/raw/bybit/perpetual/linear/trade_ticks/BTCUSDT/BTCUSDT2026-09-01.csv.gz


## Data Quality and Market Integrity

The repository should include a dedicated **data-quality and market-integrity subsystem**.

Its purpose is to answer two separate questions:

1. **Is the dataset technically reliable?**
2. **Does the recorded market activity contain suspicious statistical patterns?**

These must remain separate.

### Integrity Validation

The pipeline should validate:

- archive integrity and successful decompression
- parser completion
- required fields and numeric types
- timestamp units and precision
- instrument mapping
- exact duplicate source events
- conflicting records
- source ordering
- sequence continuity where available
- snapshot and delta consistency
- valid order-book reconstruction
- corrupted or incomplete intervals

Depth data should be validated statefully.

If an incremental book stream becomes invalid because an update is missing, the affected interval should remain untrusted until a valid resynchronization or snapshot occurs.

### Preserve Source Semantics

The validator should preserve:

- source sequence IDs
- update/message boundaries
- original timestamps
- source timestamp precision
- snapshot vs incremental semantics
- depth limits
- source event IDs

Different exchanges should not be forced into assumptions their feeds do not support.

### Suspicious Activity Analysis

The project may also calculate indicators for unusual activity, including candidate features such as:

- unusually concentrated trade sizes
- repetitive execution patterns
- unusual timing regularity
- volume relative to visible liquidity
- unusual exchange-specific volume divergence
- abnormal price/volume relationships
- distributional anomalies in trade sizes

These should be treated as **anomaly indicators**, not proof of wash trading.

The project should never claim that a dataset is guaranteed to contain no wash trading when participant identity is unavailable.

### Quality Output

Quality results should be stored separately from the market data:

```text
quality/
└── run=<audit_id>/
    ├── manifest.json
    ├── summary.parquet
    ├── findings.parquet
    ├── invalid_intervals.parquet
    ├── window_metrics.parquet
    └── report.html
```

Findings should include concepts such as:

```text
exchange
instrument
dataset
start_timestamp
end_timestamp
finding_code
severity
check_status
observed_value
reference_value
detector_version
source_event_range
```

### Do Not Use One Overall Quality Score

Prefer explicit results such as:

```text
Archive integrity:       PASS
Record validation:       PASS
Sequence continuity:     NOT EVALUABLE
Book reconstruction:     VALID WITH EXCLUDED INTERVALS
Behavioural screening:   REVIEW
```

This is more informative than a vague score such as `92/100`.

### Audit First, Filter Explicitly

The default pipeline should:

```text
AUDIT
  ↓
REPORT
  ↓
OPTIONAL FILTER POLICY
  ↓
MATERIALIZED DATASET
```

Suspicious-but-valid events should not silently disappear.

Any filtered dataset should record exactly:

- which records or intervals were removed
- which rule caused the removal
- which detector/version was used
- which source files produced the data

### Python + Rust

Rust should perform checks requiring large-scale event processing:

```text
parsing validation
duplicate checks
sequence validation
book reconstruction
ordered-stream validation
window-level metrics
```

Python should handle:

```text
audit configuration
statistical analysis
experimental anomaly detectors
report generation
quality-policy configuration
```

### Final Goal

The repository should produce not merely normalized data, but **auditable research-ready data**:

```text
Raw exchange data
        ↓
Normalize
        ↓
Validate integrity
        ↓
Validate book reconstruction
        ↓
Analyze market behaviour
        ↓
Store quality evidence
        ↓
Apply explicit research policy
        ↓
Merged Parquet
        ↓
mlfindgen
```

`mlfindgen` should therefore know not only what data it is consuming, but also **where that data is trustworthy, where quality is uncertain, and what filtering decisions were made before model research begins.**

## Market Integrity Indicators for Dataset Selection

The repository should not assume that technically valid exchange data is automatically suitable for downstream research or `mlfindgen`.

A dedicated **market-integrity analysis layer** should calculate evidence-based indicators that help determine which exchanges, instruments, dates, and intervals are appropriate for model development.

### Research-Based Indicators

The initial system should draw from published market-microstructure and crypto market-integrity research, including:

- **Benford / digit-distribution tests**
- **Trade-size roundness and clustering**
- **Trade-size distribution and tail analysis**
- **Transaction-count vs traded-volume anomalies**
- **Inter-trade timing and periodicity**
- **Entropy and complexity measures**
- **Volume vs volatility relationships**
- **Volume vs visible liquidity**
- **Trade-price impact**
- **Order-book cancellation and quote-lifetime behaviour**
- **Depth replenishment patterns**
- **Trade ↔ order-book consistency**
- **Cross-exchange price divergence**
- **Cross-exchange volume divergence**
- **Lead/lag and price-discovery relationships**

Where possible, detectors should reproduce or adapt methodologies from published research rather than rely on arbitrary thresholds.

### Multiple Indicators, Not One Detector

No single metric should determine whether data is trustworthy.

Instead:

```text
Trade statistics
       │
Order-book behaviour
       │
Cross-exchange comparisons
       │
Published integrity tests
       │
       ▼
Market-integrity evidence profile
```

An unusual result should be treated as **evidence requiring review**, not automatic proof of wash trading or manipulation.

### Dataset Selection

Integrity results should ultimately help determine what data is used for downstream development.

For example:

```text
PASS
    → suitable for normal use

PASS WITH FLAGS
    → usable, but retain integrity indicators

REVIEW
    → unusual market behaviour; inspect before ML use

INVALID
    → technical/reconstruction failure; exclude
```

This allows `mlfindgen` experiments to deliberately select:

```text
all technically valid data

high-integrity periods only

data excluding selected anomalous intervals

data including anomaly flags as features
```

### Preserve the Evidence

Quality and integrity results should be stored separately from the raw market data in Parquet/metadata so that filtering decisions remain reproducible.

Each flagged interval should record:

```text
exchange
instrument
time range
indicator
observed value
baseline
severity
method/version
research methodology/reference
```

### Goal

The repository should eventually provide not only:

> “Here is historical microstructure data.”

but:

> **“Here is historical microstructure data, together with reproducible evidence about its technical quality and observed market-integrity characteristics.”**

These indicators will later be used to decide **which datasets and periods are appropriate for development, backtesting, feature generation, and ML research in `mlfindgen`.**

## Hot-Potato Trading and Market Intermediation Analysis

The market-integrity system should distinguish suspicious/artificial activity from legitimate **hot-potato trading and market-maker intermediation**.

Hot-potato trading occurs when genuine inventory or risk is repeatedly transferred between dealers/market makers, producing substantial gross trading volume without equivalent new directional demand.

It is **not wash trading**.

### What Can Be Estimated

Without participant identities, individual hot-potato trades generally cannot be identified directly.

Instead, the repository should characterize periods along a spectrum:

```text
Directional / informational flow
              ↕
Highly intermediated / churning flow
```

### Core Indicators

Useful measures include:

- gross traded volume
- net signed order flow
- gross-volume / net-flow relationship
- trade-sign reversal frequency
- trade-sign autocorrelation
- buy/sell run lengths
- temporary price impact
- persistent price impact
- volume relative to persistent impact
- order-book depth consumption
- liquidity replenishment time
- spread recovery
- depth recovery
- cross-exchange price leadership
- cross-exchange price-discovery contribution

A simple descriptive churn statistic can be based on:

```text
1 - |net signed volume| / gross volume
```

High gross turnover combined with little net directional flow indicates substantial two-sided churn, although this alone does not establish hot-potato trading.

### Temporary vs Persistent Impact

Trades should be evaluated against future mid-price movement at several horizons:

```text
trade
  ↓
10 ms
100 ms
1 sec
5 sec
```

Large volume with mostly temporary impact suggests different market behaviour from volume associated with persistent price changes.

### Order-Book Context

Depth data should be used to measure:

- liquidity consumed by aggressive trades
- replenishment quantity
- replenishment speed
- spread recovery
- depth recovery

Large turnover combined with rapid book replenishment and little persistent price movement may indicate strong market-making/intermediation activity.

### Cross-Exchange Context

Cross-exchange merging makes the analysis substantially stronger.

For the same interval, compare:

```text
gross volume
net order flow
persistent price impact
depth consumption
price leadership
price-discovery contribution
```

across Bybit, Binance, OKX, Bitget, Gate.io, and other supported venues.

An exchange can therefore be evaluated against both:

- its own historical behaviour
- simultaneous behaviour on comparable venues

### Inter

## Price Discovery and Exchange Selection

Before deciding whether `mlfindgen` should consume one exchange or merged multi-exchange data, the repository should include a dedicated **price-discovery and venue-information analysis layer**.

The objective is to determine:

> Which exchanges actually contribute information to price formation, and does combining exchanges provide useful additional information?

### Price Discovery Analysis

For each instrument, market type, and market regime, analyze:

```text id="k7bduy"
lead / lag relationships
quote leadership
trade → trade propagation
trade → quote propagation
quote → quote propagation
quote → trade propagation
information share
component share
cross-venue price dispersion
```

Methods can eventually include established microstructure techniques such as:

- Hasbrouck Information Share
- Gonzalo–Granger Component Share
- high-frequency lead/lag analysis
- cross-correlation
- event-level information propagation

### Price Discovery ≠ Predictive Value

An exchange that usually moves first is not necessarily the only useful exchange.

For example:

```text id="r0n2ms"
Exchange B depth imbalance
        ↓
Exchange C aggressive flow
        ↓
5 ms later
        ↓
Exchange A price move
```

Exchange A may dominate measured price discovery while B and C still contain valuable predictive information.

Therefore also measure **incremental information contribution**:

> Does exchange X provide information that is not already contained in the other exchanges?

### Dynamic Leadership

Do not assume one permanent leading exchange.

Leadership may depend on:

```text id="m4ybbt"
instrument
spot vs perpetual
time of day
volatility regime
liquidity regime
market stress
news events
asset
```

The identity and strength of the current leading venue may itself become a useful feature.

### Venue Information Profile

For each venue, calculate evidence across:

```text id="e9fzk4"
PRICE DISCOVERY
    lead/lag
    information share
    quote leadership

LIQUIDITY
    spread
    depth
    replenishment
    resiliency

ORDER FLOW
    signed flow
    imbalance
    trade intensity
    aggressive flow

MARKET INTEGRITY
    technical quality
    reconstruction quality
    suspicious-activity indicators

INCREMENTAL INFORMATION
    unique information after accounting for other venues
```

### Merge Strategy Should Be Empirical

The repository should support multiple dataset constructions:

```text id="9c1y9k"
single dominant exchange

selected exchanges

all high-quality exchanges

trades-only multi-exchange merge

depth-only multi-exchange merge

complete trades + depth multi-exchange event stream
```

Normalized exchange datasets should always remain available independently.

Merged datasets are **materialized research views**, not replacements for the underlying normalized data.

### Final Validation Through `mlfindgen`

Ultimately compare:

```text id="7g57me"
Model A
    Binance only

Model B
    Binance + OKX

Model C
    Binance + OKX + Bybit

Model D
    all trusted venues
```

using proper out-of-sample evaluation.

Price-discovery analysis explains **how information moves through the market**.

Incremental-information analysis determines **whether additional venues contain unique information**.

`mlfindgen` then determines **whether that additional information actually improves the downstream models**.

The repository should therefore make both single-exchange and multi-exchange experiments inexpensive so that the final merge strategy is determined by evidence rather than assumption.